# Hachi Qwen3.5-2B Master Model V2 (All 10 Consolidated Tools & Modes)

This notebook trains the **Unified Hachi Master Brain V2** across all 10 tools:
- 🏠 Smart Home (`control_smart_home`, `get_smart_home_state`)
- 🎮 Assistant Modes (`manage_mode`: gaming, study, movie, focus)
- 📋 Multi-Step Routines (`run_routine`: daily_briefing, study_sprint, gaming_setup, research_brief)
- ⛅ Live Weather (`get_weather`: Manila, Cebu, Tokyo, London, etc.)
- 🧠 Memory & Deadlines (`manage_productivity`: notes, todos, reminders, deadlines, durable memory)
- 📸 System & Hardware (`system_control`: stats, volume, brightness, screenshots, clipboard, dictation)
- 💻 Desktop Apps (`manage_app`: launch/close Blender, VS Code, Chrome, Discord, Steam, OBS, etc.)
- 🎵 Media Playback (`media`: Spotify search/play, YouTube search/play, system playback)
- 🌐 Web Research (`web_research`: live search & synthesis)
- 💬 Conversational Chat & QA

Hardware: Tesla T4 GPU (GPU T4 x1 or x2). Precision: FP16 with PyTorch Gradient Checkpointing.


In [ ]:
# Install required dependencies
%pip install -q --upgrade unsloth unsloth_zoo datasets trl peft


In [ ]:
# Hardware and environment verification
import os, sys, json, time, re, hashlib, zipfile, torch
from pathlib import Path
from collections import defaultdict, Counter

assert torch.cuda.is_available(), 'GPU is required. Enable GPU T4 in Session Options.'
device_name = torch.cuda.get_device_name(0)
print('GPU Device:', device_name)
print('CUDA Version:', torch.version.cuda)
print('BF16 Supported:', torch.cuda.is_bf16_supported())
print('Using standard FP16 precision for Tesla T4 stability.')


In [ ]:
# Locate and extract Master V2 dataset
input_root = Path('/kaggle/input')

def valid_master_manifest(path):
    try:
        val = json.loads(path.read_text(encoding='utf-8'))
        return val.get('version') == 'master_all_in_one_3500'
    except Exception:
        return False

manifest_candidates = [p for p in input_root.rglob('manifest.json') if valid_master_manifest(p)]
if not manifest_candidates:
    zip_candidates = list(input_root.rglob('hachi-master-3500-data.zip')) + list(input_root.rglob('*master*3500*.zip'))
    if not zip_candidates:
        raise FileNotFoundError('Upload hachi-master-3500-data.zip via Add Input.')
    extract_dir = Path('/kaggle/working/hachi_training_data_master_v2')
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_candidates[0]) as z:
        z.extractall(extract_dir)
    manifest_candidates = [p for p in extract_dir.rglob('manifest.json') if valid_master_manifest(p)]

assert len(manifest_candidates) == 1, 'Expected 1 manifest file.'
DATA_DIR = manifest_candidates[0].parent
manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))

def read_jsonl(path):
    return [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines() if l.strip()]

raw_train = read_jsonl(DATA_DIR / 'train.jsonl')
raw_val = read_jsonl(DATA_DIR / 'validation.jsonl')
raw_test = read_jsonl(DATA_DIR / 'test.jsonl')

print('Loaded Master Dataset from:', DATA_DIR)
print('Split counts:', {'train': len(raw_train), 'val': len(raw_val), 'test': len(raw_test)})


In [ ]:
# Hyperparameters
SMOKE_TEST = False # Full training run
MODEL_NAME = 'Qwen/Qwen3.5-2B'
MAX_SEQ_LENGTH = 2048
LORA_RANK = 32
LORA_ALPHA = 64
LEARNING_RATE = 1e-4
MAX_EPOCHS = 6
EARLY_STOPPING_PATIENCE = 2
SEED = 9407
OUTPUT_DIR = Path('/kaggle/working/hachi-qwen35-2b-training-master-v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Training Mode:', '2-Step Smoke Test' if SMOKE_TEST else 'FULL ALL-IN-ONE V2 TRAINING RUN')


In [ ]:
# Load Qwen3.5-2B base model with native FP16 precision
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import Dataset
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)
model = model.to(torch.float16)
print('Loaded base model in FP16:', MODEL_NAME)


In [ ]:
# Format chat template
def format_record(record):
    return tokenizer.apply_chat_template(
        record['messages'], tools=record['tools'], tokenize=False, add_generation_prompt=False
    )

train_texts = [format_record(r) for r in raw_train]
val_texts = [format_record(r) for r in raw_val]
train_dataset = Dataset.from_dict({'text': train_texts})
val_dataset = Dataset.from_dict({'text': val_texts})

lengths = [len(tokenizer(text=t, add_special_tokens=False)['input_ids']) for t in train_texts + val_texts]
print('Token lengths: min', min(lengths), 'median', sorted(lengths)[len(lengths)//2], 'max', max(lengths))
assert max(lengths) <= MAX_SEQ_LENGTH, 'Record exceeds MAX_SEQ_LENGTH'


In [ ]:
# Apply LoRA with standard PyTorch gradient checkpointing
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing=True,
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

common_args = dict(
    output_dir=str(OUTPUT_DIR), max_seq_length=MAX_SEQ_LENGTH, dataset_text_field='text',
    per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=LEARNING_RATE, warmup_steps=5, weight_decay=0.01, optim='adamw_8bit',
    logging_steps=1 if SMOKE_TEST else 5, report_to='none', seed=SEED, dataset_num_proc=1,
    fp16=True, bf16=False, packing=False,
)
if SMOKE_TEST:
    run_args = dict(max_steps=2, eval_strategy='steps', eval_steps=1, save_strategy='no')
else:
    run_args = dict(
        num_train_epochs=MAX_EPOCHS, eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
        save_total_limit=2,
    )

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_dataset, eval_dataset=val_dataset,
    args=SFTConfig(**common_args, **run_args),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
    force_match=True,
)
print('Trainer ready with response-only masking.')


In [ ]:
"""Strict, offline scoring of Hachi model outputs; never executes tools.

Tool accuracy measures selection and arguments, not application execution.
Conversational quality requires a separate human rubric and remains unscored.
The notebook embeds these helpers via scripts/sync_master_evaluator.py.
"""
import json
import re


SCORER_VERSION = "strict_v1"
VALID_TOOLS = {
    "control_smart_home", "get_smart_home_state", "manage_app", "manage_mode",
    "run_routine", "media", "manage_productivity", "web_research", "get_weather",
    "system_control",
}


def parse_prediction(raw):
    """Parse every call, rejecting incomplete syntax, unknown tools and duplicates."""
    text = re.sub(r"(?:<\|im_end\|>|<\|endoftext\|>|<\|eot_id\|>)\s*$", "", str(raw or "")).strip()
    calls = []

    def json_call(value):
        if not isinstance(value, dict):
            raise ValueError("Tool call must be an object")
        fn = value.get("function", value)
        if not isinstance(fn, dict):
            raise ValueError("Function must be an object")
        args = fn.get("arguments", {})
        if isinstance(args, str):
            args = json.loads(args)
        return {"name": fn.get("name"), "arguments": args}

    def parse_block(block):
        function = re.fullmatch(r"<function=([\w]+)>\s*(.*?)\s*</function>", block.strip(), re.DOTALL)
        if not function:
            return json_call(json.loads(block))
        name, body = function.groups()
        args = {}
        pattern = r"<parameter=([\w]+)>\s*(.*?)\s*</parameter>"
        for match in re.finditer(pattern, body, re.DOTALL):
            key, value = match.groups()
            if key in args:
                raise ValueError("Duplicate parameter")
            try:
                args[key] = json.loads(value)
            except json.JSONDecodeError:
                args[key] = value.strip()
        if re.sub(pattern, "", body, flags=re.DOTALL).strip():
            raise ValueError("Malformed function parameters")
        return {"name": name, "arguments": args}

    try:
        if "<tool_call>" in text:
            blocks = re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", text, re.DOTALL)
            remainder = re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.DOTALL).strip()
            if not blocks or remainder:
                raise ValueError("Incomplete or mixed tool-call output")
            calls = [parse_block(block) for block in blocks]
        elif "<function=" in text:
            calls = [parse_block(text)]
        elif any(marker in text for marker in ("tool_call", "<parameter=", "</function>", '"arguments"')):
            value = json.loads(text)
            values = value.get("tool_calls", [value]) if isinstance(value, dict) else value
            if not isinstance(values, list) or not values:
                raise ValueError("Missing tool calls")
            calls = [json_call(value) for value in values]
        elif text.startswith("{"):
            try:
                value = json.loads(text)
            except json.JSONDecodeError:
                value = None
            if isinstance(value, dict) and ("name" in value or "function" in value):
                calls = [json_call(value)]
        for call in calls:
            if call["name"] not in VALID_TOOLS:
                raise ValueError("Unknown tool")
            if not isinstance(call["arguments"], dict):
                raise ValueError("Arguments must be an object")
        return {"calls": calls, "error": None, "text": text}
    except (ValueError, TypeError, AttributeError) as exc:
        return {"calls": calls, "error": str(exc), "text": text}


def score_output(expected, raw):
    parsed = parse_prediction(raw)
    calls = parsed["calls"]
    no_tool = not calls and parsed["error"] is None and bool(parsed["text"])
    if expected["outcome"] == "assistant_response":
        return {**parsed, "correct": None, "no_tool_compliant": no_tool,
                "quality_review_required": True}
    correct = parsed["error"] is None and len(calls) == 1
    if correct:
        call = calls[0]
        wanted = dict(expected.get("arguments", {}))
        actual = dict(call["arguments"])
        if expected["tool"] == "control_smart_home":
            # Goal prose is descriptive; action/value/target accuracy is exact.
            wanted.pop("goal", None)
            actual.pop("goal", None)
            for args in (wanted, actual):
                if isinstance(args.get("actions"), list):
                    args["actions"] = sorted(json.dumps(a, sort_keys=True) for a in args["actions"])
        correct = call["name"] == expected["tool"] and actual == wanted
    return {**parsed, "correct": bool(correct), "no_tool_compliant": None,
            "quality_review_required": False}


def summarize_scores(rows):
    tool_rows = [r for r in rows if r["expected"]["outcome"] == "tool_call"]
    chat_rows = [r for r in rows if r["expected"]["outcome"] == "assistant_response"]
    return {
        "scorer_version": SCORER_VERSION,
        "records": len(rows),
        "tool_call_records": len(tool_rows),
        "tool_call_correct": sum(r["correct"] is True for r in tool_rows),
        "tool_call_accuracy": sum(r["correct"] is True for r in tool_rows) / len(tool_rows) if tool_rows else None,
        "assistant_response_records": len(chat_rows),
        "no_tool_compliance": sum(r["no_tool_compliant"] for r in chat_rows) / len(chat_rows) if chat_rows else None,
        "invalid_output_records": sum(r["error"] is not None for r in rows),
        "response_quality_accuracy": None,
        "end_to_end_task_accuracy": None,
        "limitations": ["Conversational quality is unscored and requires human review.",
                        "Tool outputs are scored without executing application actions.",
                        "No laptop latency or GGUF quality is inferred from training runs."],
    }




@torch.inference_mode()
def evaluate_records(records, label):
    FastLanguageModel.for_inference(model)
    results, started = [], time.perf_counter()
    text_tok = tokenizer.tokenizer if hasattr(tokenizer, 'tokenizer') else tokenizer
    eos_ids = [text_tok.eos_token_id]
    im_end_id = text_tok.convert_tokens_to_ids('<|im_end|>')
    if isinstance(im_end_id, int) and im_end_id >= 0 and im_end_id not in eos_ids:
        eos_ids.append(im_end_id)
    for number, record in enumerate(records, 1):
        prompt = tokenizer.apply_chat_template(
            record['messages'][:2], tools=record['tools'], tokenize=False, add_generation_prompt=True
        )
        encoded = tokenizer(text=prompt, return_tensors='pt', add_special_tokens=False).to(model.device)
        generated = model.generate(
            **encoded, max_new_tokens=160, do_sample=False, use_cache=True,
            eos_token_id=eos_ids, pad_token_id=text_tok.eos_token_id,
        )
        completion = text_tok.decode(
            generated[0][encoded['input_ids'].shape[1]:], skip_special_tokens=False
        )
        score = score_output(record['expected'], completion)
        correct = score['correct']
        results.append({
            'id': record['id'], 'category': record['category'], 'language': record['language'],
            'expected': record['expected'], **score,
            'raw': completion
        })
        print(f'[{number}/{len(records)}] {record["id"]}: {correct}')
    elapsed = time.perf_counter() - started
    metrics = {
        'label': label, 'records': len(results),
        **summarize_scores(results),
        'latency_environment': 'training_notebook',
        'seconds': elapsed, 'seconds_per_record': elapsed / len(results)
    }
    print(json.dumps(metrics, indent=2))
    return metrics, results


In [ ]:
# Baseline Evaluation
eval_records = raw_test[:10] if SMOKE_TEST else raw_test
base_metrics, base_results = evaluate_records(eval_records, 'pre_training_master_v2_base')
FastLanguageModel.for_training(model)


In [ ]:
# Train Model
torch.cuda.reset_peak_memory_stats()
train_start = time.perf_counter()
train_res = trainer.train()
train_secs = time.perf_counter() - train_start
peak_gib = torch.cuda.max_memory_allocated() / (1024**3)
print('Training Completed in', round(train_secs, 1), 'seconds')
print('Peak Allocated VRAM:', round(peak_gib, 2), 'GiB')


In [ ]:
# Post-Training Evaluation
post_metrics, post_results = evaluate_records(eval_records, 'post_training_master_v2')
eval_payload = {
    'experiment': 'master_all_in_one_3500',
    'baseline': base_metrics,
    'baseline_results': base_results,
    'post_training': post_metrics,
    'results': post_results
}
(OUTPUT_DIR / 'evaluation_results_master_v2.json').write_text(json.dumps(eval_payload, indent=2), encoding='utf-8')
print('Evaluation results saved to evaluation_results_master_v2.json')


In [ ]:
# GGUF Export & Quantization
if not SMOKE_TEST:
    gguf_dir = OUTPUT_DIR / 'hachi-master-v2-qwen35-2b-q4_k_m'
    model.save_pretrained_gguf(str(gguf_dir), tokenizer, quantization_method='q4_k_m')
    zip_path = Path('/kaggle/working/hachi-master-v2-qwen35-2b-q4_k_m-GGUF.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in gguf_dir.rglob('*'):
            if f.is_file():
                z.write(f, f.relative_to(gguf_dir.parent))
    print('Exported Quantized GGUF Archive:', zip_path)
else:
    print('SMOKE TEST COMPLETE - Ready for Full Run!')
